# Workflow Evaluation - Metrics by Category

Standalone analysis of `evaluation_results/eval_*.csv` (latest export).

**Data:** 175 evaluation cases (5 stocks x 5 categories x 7 prompt styles) x 3 workflows (A/B/C) = 525 rows.
Each workflow ran the identical 175 cases, so workflow comparisons are **paired**.

Comparisons:
- Workflow A (`agentic_workflow_A_bert`)
- Workflow B (`agentic_workflow_B_finbert`)
- Workflow C (`agentic_workflow_C_no_sentiment`)

The four deepeval metrics (all judged with threshold 0.5):
- **TaskCompletion** - did the final report complete the request
- **ArgumentCorrectness** - were tool-call arguments sensible
- **AnswerRelevancy** - does the report answer the prompt
- **Hallucination** - is the report grounded in gathered context

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT = Path().resolve().parent
# analysis always reads the latest exported eval CSV (timestamped filenames sort chronologically)
_eval_csvs = sorted((PROJECT / "evaluation_results").glob("eval_*.csv"))
assert _eval_csvs, ("no eval CSV in evaluation_results yet - activate after the first v2 export")
CSV_PATH = _eval_csvs[-1]

WORKFLOWS = ["A", "B", "C"]
METRICS = ["TaskCompletionMetric", "ArgumentCorrectnessMetric",
           "AnswerRelevancyMetric", "HallucinationMetric"]
METRIC_LABELS = ["Task Completion", "Argument Correctness",
                 "Answer Relevancy", "Hallucination"]
SHORT = ["TC", "AC", "AR", "HALL"]

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print(f"CSV: {CSV_PATH}")

**Data note:** the row-level `success` column in the CSV means *all four evaluators
completed without error*, not overall quality — it is saturated (100%) since the
2026-10-28 retry sweep and is intentionally ignored here. The quality flag used
alongside raw scores is each metric's `passed` (judge score >= 0.5).

## 1. Load & validate

In [ ]:
df = pd.read_csv(CSV_PATH)

for m in METRICS:
    df[f"{m}_passed_bool"] = df[f"{m}_passed"].astype(bool)

# derive analysis dimensions from the data (tolerates future prompt matrices)
CATEGORIES = [c for c in df["category"].unique() if isinstance(c, str)]
STOCKS = sorted(df["stock"].dropna().unique())
STYLES = [s for s in df["prompt_style"].unique() if isinstance(s, str)]

print(df.shape)
assert set(df["workflow"].unique()) == set(WORKFLOWS)

# every (prompt_id) must appear exactly once per workflow
counts = df.groupby("prompt_id")["workflow"].nunique()
assert (counts == 3).all(), "some cases are missing a workflow run"
assert df.duplicated(["prompt_id", "workflow"]).sum() == 0

n_cases = df["prompt_id"].nunique()
print(f"{n_cases} cases x {len(WORKFLOWS)} workflows = {len(df)} rows")
print(f"categories: {CATEGORIES}")
print(f"stocks: {STOCKS}")
print(f"styles: {STYLES}")
for m in METRICS:
    assert df[f"{m}_score"].notna().all(), f"missing scores in {m} - rerun the retry cell"
print("all metric scores complete")

## 2. Headline: overall workflow comparison

Overall mean score per metric, per workflow. Runs are paired (identical 175 cases),
so mean differences are directly attributable to the workflow.

In [ ]:
overall = df.groupby("workflow")[[f"{m}_score" for m in METRICS]].mean()
overall = overall.rename(columns={f"{m}_score": lbl for m, lbl in zip(METRICS, METRIC_LABELS)})
overall = overall.reindex(WORKFLOWS)

# pass rates (judged threshold 0.5) replace the old saturated 'success' column
for m, lbl in zip(METRICS, SHORT):
    overall[f"{lbl} pass %"] = df.groupby("workflow")[f"{m}_passed_bool"].mean().reindex(WORKFLOWS) * 100

display(overall.round(3))

### Overall distribution per metric

One boxplot subplot per metric; x-axis = workflow.

In [ ]:
fig, axes = plt.subplots(1, len(METRICS), figsize=(4 * len(METRICS), 4.2), sharey=True)

for ax, metric, label in zip(axes, METRICS, METRIC_LABELS):
    data = [df.loc[df["workflow"] == w, f"{metric}_score"] for w in WORKFLOWS]
    bp = ax.boxplot(data, tick_labels=WORKFLOWS, showmeans=True, patch_artist=True,
                    meanprops={"marker": "D", "markerfacecolor": "white",
                               "markeredgecolor": "black", "markersize": 4})
    for box in bp["boxes"]:
        box.set(facecolor="#a9c7f0", alpha=0.7)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Workflow")
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("Metric score (0-1)")
fig.suptitle("Overall score distribution per workflow", fontsize=13)
fig.tight_layout()
fig.savefig(PROJECT / "reports" / "eval_boxplots_overall.png", bbox_inches="tight")
plt.show()

## 3. Per-category analysis (main)

Mean metric scores per category x workflow, with the best workflow per category.

In [ ]:
def pivot_table_best(metric, index):
    tbl = (df.pivot_table(index=index, columns="workflow",
                          values=f"{metric}_score", aggfunc="mean")
             .reindex(columns=WORKFLOWS))
    tbl["best"] = tbl.idxmax(axis=1)
    return tbl


for metric, label in zip(METRICS, METRIC_LABELS):
    print(f"--- {label}: mean score by category x workflow ---")
    display(pivot_table_best(metric, "category").reindex(CATEGORIES).round(3))
    print()

### Boxplots: category x workflow, one subplot per metric

In [ ]:
fig, axes = plt.subplots(len(CATEGORIES), len(METRICS),
                         figsize=(3.4 * len(METRICS), 2.6 * len(CATEGORIES)),
                         sharex=True)

for i, cat in enumerate(CATEGORIES):
    sub = df[df["category"] == cat]
    for j, (metric, label) in enumerate(zip(METRICS, METRIC_LABELS)):
        ax = axes[i, j]
        data = [sub.loc[sub["workflow"] == w, f"{metric}_score"] for w in WORKFLOWS]
        bp = ax.boxplot(data, tick_labels=WORKFLOWS, showmeans=True, patch_artist=True,
                        meanprops={"marker": "D", "markerfacecolor": "white",
                                   "markeredgecolor": "black", "markersize": 3})
        for box in bp["boxes"]:
            box.set(facecolor="#a9c7f0", alpha=0.7)
        ax.grid(axis="y", alpha=0.3)
        if j == 0:
            ax.set_ylabel(cat, fontsize=9)
        if i == 0:
            ax.set_title(label, fontsize=11)

fig.suptitle("Score distribution: category (rows) x metric (cols), x = workflow",
             fontsize=13)
fig.tight_layout()
fig.savefig(PROJECT / "reports" / "eval_boxplots_by_category.png", bbox_inches="tight")
plt.show()

## 4. Prompt-style analysis

Same comparison split by the 7 prompt styles of the test matrix.
Each style x workflow cell averages 25 paired cases.

In [ ]:
for metric, label in zip(METRICS, METRIC_LABELS):
    print(f"--- {label}: mean score by prompt style x workflow ---")
    display(pivot_table_best(metric, "prompt_style").reindex(STYLES).round(3))
    print()

### Small-multiple heatmaps: prompt style x workflow, one per metric

In [ ]:
fig, axes = plt.subplots(1, len(METRICS),
                         figsize=(2.6 * len(METRICS), 0.5 * len(STYLES) + 1.2))

for ax, metric, label in zip(axes, METRICS, METRIC_LABELS):
    mat = (df.pivot_table(index="prompt_style", columns="workflow",
                          values=f"{metric}_score", aggfunc="mean")
             .reindex(index=STYLES, columns=WORKFLOWS))
    im = ax.imshow(mat.values, cmap="RdYlBu", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(WORKFLOWS)), WORKFLOWS, fontsize=8)
    ax.set_yticks(range(len(STYLES)), STYLES, fontsize=8)
    for i in range(len(STYLES)):
        for j in range(len(WORKFLOWS)):
            ax.text(j, i, f"{mat.values[i, j]:.2f}", ha="center", va="center",
                    fontsize=7)
    ax.set_title(label, fontsize=10)

fig.colorbar(im, ax=axes, shrink=0.8, label="mean score")
fig.suptitle("Mean metric score: prompt style (rows) x workflow (cols)", fontsize=12)
fig.savefig(PROJECT / "reports" / "eval_scores_by_prompt_style.png", bbox_inches="tight")
plt.show()

### Boxplots: prompt style x workflow, one subplot per metric

In [ ]:
fig, axes = plt.subplots(len(STYLES), len(METRICS),
                         figsize=(3.4 * len(METRICS), 2.0 * len(STYLES)),
                         sharex=True)

for i, style in enumerate(STYLES):
    sub = df[df["prompt_style"] == style]
    for j, (metric, label) in enumerate(zip(METRICS, METRIC_LABELS)):
        ax = axes[i, j]
        data = [sub.loc[sub["workflow"] == w, f"{metric}_score"] for w in WORKFLOWS]
        bp = ax.boxplot(data, tick_labels=WORKFLOWS, showmeans=True, patch_artist=True,
                        meanprops={"marker": "D", "markerfacecolor": "white",
                                   "markeredgecolor": "black", "markersize": 3})
        for box in bp["boxes"]:
            box.set(facecolor="#a9c7f0", alpha=0.7)
        ax.grid(axis="y", alpha=0.3)
        if j == 0:
            ax.set_ylabel(style, fontsize=9)
        if i == 0:
            ax.set_title(label, fontsize=11)

fig.suptitle("Score distribution: prompt style (rows) x metric (cols), x = workflow",
             fontsize=13)
fig.tight_layout()
fig.savefig(PROJECT / "reports" / "eval_boxplots_by_prompt_style.png", bbox_inches="tight")
plt.show()

### Best workflow per prompt style

Same summary pattern as the category breakdown.

In [ ]:
styles_summary = pd.DataFrame({
    m: pivot_table_best(metric, "prompt_style").reindex(STYLES)["best"]
    for m, metric in zip(SHORT, METRICS)
})
display(styles_summary)

## 5. Per-stock drill-down

Mean TaskCompletion score for every stock x category x prompt_style cell, per workflow
(175 cells). `best` names the workflow with the highest TaskCompletion score in that cell.

In [ ]:
piv = df.pivot_table(index=["stock", "category", "prompt_style"], columns="workflow",
                     values="TaskCompletionMetric_score", aggfunc="mean")
piv["best"] = piv[WORKFLOWS].idxmax(axis=1)
display(piv.reindex(STOCKS, level="stock").round(3))

print("Cells won (TaskCompletion, of 175):")
print(piv[WORKFLOWS].idxmax(axis=1).value_counts())

## 6. Workflow runtime (only for runs that recorded it)

`run_seconds` = per-case wall time of the workflow (agent + internal LLM calls).
Legacy v1 runs have no timing, so this section activates only if the current CSV
carries the field.

In [ ]:
if "run_seconds" in df.columns and df["run_seconds"].notna().any():
    rt = df.groupby("workflow")["run_seconds"].agg(mean="mean", median="median",
                                                  min="min", max="max", n="count")
    display(rt.round(1))

    fig, ax = plt.subplots(figsize=(5.5, 4.2))
    bp = ax.boxplot([df.loc[df["workflow"] == w, "run_seconds"].dropna()
                     for w in WORKFLOWS],
                    tick_labels=WORKFLOWS, showmeans=True, patch_artist=True)
    for box in bp["boxes"]:
        box.set(facecolor="#a9c7f0", alpha=0.7)
    ax.set_ylabel("run_seconds (per case)")
    ax.set_title("Workflow runtime")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(PROJECT / "reports" / "eval_runtime_by_workflow.png", bbox_inches="tight")
    plt.show()
else:
    print("no run_seconds present in this CSV yet - section activates on timed runs (v2 matrix)")

## 7. Summary: best workflow per category

Rule for "best": whichever workflow has the highest **TaskCompletion** mean score in
that category; the TaskCompletion pass-rate (judge threshold 0.5) is shown alongside
as a binary check, since scores can be close.

In [ ]:
tc_tbl = pivot_table_best("TaskCompletionMetric", "category").reindex(CATEGORIES)
per_wf_pass = (df.groupby(["category", "workflow"])["TaskCompletionMetric_passed_bool"].mean()
                 .unstack().reindex(index=CATEGORIES, columns=WORKFLOWS) * 100)

summary = pd.DataFrame({
    "best_tc_workflow": tc_tbl["best"],
    **{f"pass_{w}_%": per_wf_pass[w] for w in WORKFLOWS},
})
display(summary.round(1))

print()
print("Overall best workflow (TaskCompletion):",
      overall["Task Completion"].idxmax())